# 08 — Critic Agent

Tests `agents/critic_agent.py`:
- `critique()` returns the right schema (approved, issues, suggestion)
- Approves a good recommendation
- Rejects a recommendation that violates allergen constraints
- Handles missing profile gracefully
- Issues list is populated on rejection

> **Requires**: LLM credentials in `.env`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

In [ ]:
from agents.critic_agent import critique

## Helper data

In [ ]:
SAMPLE_PRODUCTS = [
    {'name': 'Vanilla Birthday Cake', 'price': 'RS.2500', 'category': 'cakes',
     'description': 'A classic vanilla sponge cake with cream frosting. No nuts.', 'availability': 'In Stock'},
    {'name': 'Red Rose Bouquet', 'price': 'RS.1800', 'category': 'flowers',
     'description': '12 fresh red roses arranged beautifully.', 'availability': 'In Stock'},
    {'name': 'Dark Chocolate Truffle Cake', 'price': 'RS.3200', 'category': 'cakes',
     'description': 'Rich dark chocolate cake with walnut topping.', 'availability': 'In Stock'},
]

GOOD_RECOMMENDATION = """
Based on your wife's preferences, I recommend the Vanilla Birthday Cake (RS.2500). 
It's a classic vanilla sponge with cream frosting — no nuts or dairy allergens. 
This is currently in stock and can be delivered to Colombo tomorrow.
"""

BAD_RECOMMENDATION_WITH_NUTS = """
I recommend the Dark Chocolate Truffle Cake with walnut topping (RS.3200). 
It's rich and decadent, perfect for any occasion!
"""

PROFILE_WITH_NUT_ALLERGY = {
    'allergies': ['nuts', 'walnuts'],
    'preferences': ['vanilla'],
    'location': 'Colombo'
}

## 1. Return schema validation

In [ ]:
result = critique(
    recommendation=GOOD_RECOMMENDATION,
    search_query='birthday cake for wife',
    profile=PROFILE_WITH_NUT_ALLERGY,
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Critique result:', result)

assert isinstance(result, dict), 'critique() must return a dict'
assert 'approved' in result, 'Result must have "approved" key'
assert 'issues' in result, 'Result must have "issues" key'
assert 'suggestion' in result, 'Result must have "suggestion" key'
assert isinstance(result['approved'], bool), '"approved" must be bool'
assert isinstance(result['issues'], list), '"issues" must be list'
print('Schema validation: PASSED')

## 2. Approves a good recommendation (no allergen violation)

In [ ]:
result = critique(
    recommendation=GOOD_RECOMMENDATION,
    search_query='birthday cake for wife',
    profile=PROFILE_WITH_NUT_ALLERGY,
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Approved:', result['approved'])
print('Issues  :', result['issues'])

assert result['approved'] is True, \
    f'Good recommendation should be approved. Issues: {result["issues"]}'
print('Approves good recommendation: PASSED')

## 3. Rejects a recommendation with allergen violation

In [ ]:
result = critique(
    recommendation=BAD_RECOMMENDATION_WITH_NUTS,
    search_query='birthday cake for wife',
    profile=PROFILE_WITH_NUT_ALLERGY,
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Approved  :', result['approved'])
print('Issues    :', result['issues'])
print('Suggestion:', result['suggestion'])

assert result['approved'] is False, \
    'Nut-containing recommendation should be rejected for nut-allergic recipient'
assert len(result['issues']) > 0, 'Should report at least one issue'
print('Rejects allergen violation: PASSED')

## 4. Suggestion is not None on rejection

In [ ]:
result = critique(
    recommendation=BAD_RECOMMENDATION_WITH_NUTS,
    search_query='birthday cake for wife',
    profile=PROFILE_WITH_NUT_ALLERGY,
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Suggestion:', result['suggestion'])

if not result['approved']:
    assert result['suggestion'] is not None, \
        'Rejected recommendations should include a suggestion for revision'
    assert len(result['suggestion']) > 10, 'Suggestion should be non-trivial'

print('Suggestion present on rejection: PASSED')

## 5. Empty profile (no constraints)

In [ ]:
result = critique(
    recommendation=GOOD_RECOMMENDATION,
    search_query='gift for wife',
    profile={'allergies': [], 'preferences': [], 'location': ''},
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Empty profile result:', result)
assert isinstance(result, dict)
assert 'approved' in result
print('Empty profile: PASSED')

## 6. Multiple issues on a bad recommendation

In [ ]:
very_bad_recommendation = """
I recommend a mystery gift box. It may or may not contain nuts and dairy products. 
The price is unknown and availability is unclear.
"""

result = critique(
    recommendation=very_bad_recommendation,
    search_query='birthday cake for wife',
    profile=PROFILE_WITH_NUT_ALLERGY,
    recipients={'wife'},
    products=SAMPLE_PRODUCTS,
)

print('Issues:', result['issues'])
print('Approved:', result['approved'])

assert result['approved'] is False
# Should flag multiple issues (allergen + accuracy + quality)
print(f'Number of issues: {len(result["issues"])}')
print('Multiple issues test: PASSED')